<a href="https://colab.research.google.com/github/LucianoBV/Procesamiento-del-habla/blob/main/Luciano_Vargas_TP5_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP4 Chatbots basados en recuperación de la información.

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [1]:
!pip install spacy --quiet
!python -m spacy download es_core_news_md --quiet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 18.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado eligió.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





### Alumno: Vargas Luciano

# 1) Dataset de preguntas y respuestas

## Aplicación elegida

Se desarrollará un chatbot orientado a la atención al cliente de un supermercado.

El objetivo del chatbot será responder preguntas frecuentes relacionadas con horarios, pagos, envíos, cambios de productos, promociones y facturación.

Este tipo de chatbot puede ayudar a automatizar consultas simples y mejorar la atención al cliente.

In [2]:
preguntas_respuestas = [
    ("¿Cuál es el horario del supermercado?", "El supermercado abre de 8:00 a 22:00 horas."),

    ("¿Aceptan tarjetas de crédito?", "Sí, aceptamos todos los medios de pago."),

    ("¿Que medios de pago aceptan?", "Aceptamos todos los medios de pago."),

    ("¿Tienen envío a domicilio?", "Sí, realizamos envíos a domicilio dentro de la ciudad."),

    ("¿Cómo puedo hacer un reclamo?", "Puede realizar un reclamo en atención al cliente o por teléfono."),

    ("¿Dónde puedo consultar promociones?", "Las promociones están disponibles en nuestra página web."),

    ("¿Puedo cambiar un producto?", "Sí, puede cambiar productos con ticket de compra."),

    ("¿Aceptan Mercado Pago?", "Sí, aceptamos Mercado Pago."),

    ("¿Cuál es el teléfono del supermercado?", "El teléfono es 351-1234567."),

    ("¿Tienen atención los domingos?", "Sí, abrimos los domingos."),

    ("¿Venden productos congelados?", "Sí, contamos con una sección de congelados."),

    ("¿Dónde está atención al cliente?", "Atención al cliente se encuentra cerca de las cajas."),

    ("¿Puedo pagar en cuotas?", "Sí, algunas tarjetas tienen promociones en cuotas."),

    ("¿Tienen estacionamiento?", "Sí, el supermercado posee estacionamiento gratuito."),

    ("¿Cómo solicito factura A?", "Tiene que inscribirse en Atencion al cliente primero."),

    ("¿Hay promociones bancarias?", "Sí, trabajamos con promociones de distintos bancos."),

    ("¿Tienen carnicería?", "Sí, contamos con ditintos sectores."),

    ("¿Puedo devolver un producto?", "Sí, puede devolver productos según las políticas del local."),

    ("¿Aceptan billeteras virtuales?", "Sí, aceptamos distintas billeteras virtuales."),

    ("¿Dónde están las ofertas?", "Las ofertas están señalizadas dentro del local."),

    ("¿Tienen panadería?", "Sí, contamos con panadería propia.")
]

# 2) Chatbot usando TF-IDF y similitud del coseno

En esta etapa se construye un chatbot basado en recuperación de información.

El funcionamiento consiste en comparar la pregunta ingresada por el usuario con todas las preguntas del dataset. Para eso se utiliza TF-IDF, que transforma las frases en vectores numéricos, y luego se aplica similitud del coseno para medir qué tan parecidas son las preguntas.

Si la similitud es alta, el chatbot devuelve la respuesta correspondiente. Si la similitud es baja, responde que no entendió la consulta.

In [3]:
# Separamos las preguntas y respuestas del dataset
preguntas = [item[0] for item in preguntas_respuestas]
respuestas = [item[1] for item in preguntas_respuestas]

# Creamos el vectorizador TF-IDF
vectorizador = TfidfVectorizer()

# Convertimos las preguntas del dataset en vectores TF-IDF
matriz_tfidf = vectorizador.fit_transform(preguntas)

def chatbot_tfidf(pregunta_usuario):
    # Convertimos la pregunta del usuario al mismo formato TF-IDF
    vector_pregunta = vectorizador.transform([pregunta_usuario])

    # Calculamos la similitud del coseno entre la pregunta del usuario y las preguntas del dataset
    similitudes = cosine_similarity(vector_pregunta, matriz_tfidf)

    # Buscamos el índice de la pregunta más parecida
    indice_mas_parecido = similitudes.argmax()

    # Obtenemos el valor de similitud más alto
    similitud_maxima = similitudes[0][indice_mas_parecido]

    # Si la similitud es baja, respondemos que no se entendió la consulta
    if similitud_maxima <  0.6:
        return "No entendí tu consulta. Por favor, realizá una pregunta relacionada con el supermercado."

    # Si la similitud es suficiente, devolvemos la respuesta correspondiente
    return respuestas[indice_mas_parecido]

In [4]:
preguntas_prueba = [
    "¿Aceptan tarjetas?",
    "¿Cuál es el horario?",
    "¿Tienen estacionamiento?",
    "¿Hay envío a domicilio?",

    # Preguntas parcialmente relacionadas
    "¿Puedo reservar productos?",
    "¿Tienen farmacia?",
    "¿Hacen entregas fuera de la ciudad?",
    "¿Se pueden reparar electrodomésticos?",

    # Preguntas totalmente incorrectas
    "¿Cuál es el clima hoy?",
    "¿Me pueden dar fiado?",
    "¿Hola, como me llamo?",
    "¿Cómo puedo contactarte?",
]

print("PRUEBAS CHATBOT TF-IDF\n")

for pregunta in preguntas_prueba:
    respuesta = chatbot_tfidf(pregunta)
    print("Pregunta:", pregunta)
    print("Respuesta:", respuesta)
    print()

PRUEBAS CHATBOT TF-IDF

Pregunta: ¿Aceptan tarjetas?
Respuesta: Sí, aceptamos todos los medios de pago.

Pregunta: ¿Cuál es el horario?
Respuesta: El supermercado abre de 8:00 a 22:00 horas.

Pregunta: ¿Tienen estacionamiento?
Respuesta: Sí, el supermercado posee estacionamiento gratuito.

Pregunta: ¿Hay envío a domicilio?
Respuesta: Sí, realizamos envíos a domicilio dentro de la ciudad.

Pregunta: ¿Puedo reservar productos?
Respuesta: No entendí tu consulta. Por favor, realizá una pregunta relacionada con el supermercado.

Pregunta: ¿Tienen farmacia?
Respuesta: No entendí tu consulta. Por favor, realizá una pregunta relacionada con el supermercado.

Pregunta: ¿Hacen entregas fuera de la ciudad?
Respuesta: No entendí tu consulta. Por favor, realizá una pregunta relacionada con el supermercado.

Pregunta: ¿Se pueden reparar electrodomésticos?
Respuesta: No entendí tu consulta. Por favor, realizá una pregunta relacionada con el supermercado.

Pregunta: ¿Cuál es el clima hoy?
Respuesta: E

# 3) Chatbot utilizando embeddings

En esta etapa se construye un segundo chatbot utilizando embeddings pre-entrenados.

Para este trabajo se eligió el modelo es_core_news_md de la librería spaCy, ya que trabaja con idioma español y contiene vectores pre-entrenados.

A diferencia de TF-IDF, los embeddings permiten representar las frases según su significado semántico. Esto permite que el chatbot pueda relacionar consultas escritas de manera distinta, aunque no usen exactamente las mismas palabras que las preguntas del dataset.

In [5]:
# Cargamos el modelo de español con vectores pre-entrenados
nlp = spacy.load("es_core_news_md")

# Separamos preguntas y respuestas del dataset
preguntas = [item[0] for item in preguntas_respuestas]
respuestas = [item[1] for item in preguntas_respuestas]

# Convertimos cada pregunta del dataset en un vector
vectores_preguntas = np.array([nlp(pregunta).vector for pregunta in preguntas])

def chatbot_embeddings(pregunta_usuario):
    # Convertimos la pregunta del usuario en vector
    vector_usuario = nlp(pregunta_usuario).vector.reshape(1, -1)

    # Calculamos la similitud del coseno entre la pregunta del usuario y las preguntas del dataset
    similitudes = cosine_similarity(vector_usuario, vectores_preguntas)

    # Buscamos la pregunta más parecida
    indice_mas_parecido = similitudes.argmax()

    # Obtenemos la similitud más alta
    similitud_maxima = similitudes[0][indice_mas_parecido]

    # Si la similitud es baja, respondemos que la consulta no corresponde
    if similitud_maxima < 0.7:
        return "No entendí tu consulta. Por favor, realizá una pregunta relacionada con el supermercado."

    # Si la similitud es suficiente, devolvemos la respuesta correspondiente
    return respuestas[indice_mas_parecido]

In [6]:
print("PRUEBAS CHATBOT EMBEDDINGS\n")

for pregunta in preguntas_prueba:
    respuesta = chatbot_embeddings(pregunta)
    print("Pregunta:", pregunta)
    print("Respuesta:", respuesta)
    print()

PRUEBAS CHATBOT EMBEDDINGS

Pregunta: ¿Aceptan tarjetas?
Respuesta: Sí, aceptamos distintas billeteras virtuales.

Pregunta: ¿Cuál es el horario?
Respuesta: El supermercado abre de 8:00 a 22:00 horas.

Pregunta: ¿Tienen estacionamiento?
Respuesta: Sí, el supermercado posee estacionamiento gratuito.

Pregunta: ¿Hay envío a domicilio?
Respuesta: Sí, realizamos envíos a domicilio dentro de la ciudad.

Pregunta: ¿Puedo reservar productos?
Respuesta: Las promociones están disponibles en nuestra página web.

Pregunta: ¿Tienen farmacia?
Respuesta: Sí, contamos con ditintos sectores.

Pregunta: ¿Hacen entregas fuera de la ciudad?
Respuesta: Sí, aceptamos todos los medios de pago.

Pregunta: ¿Se pueden reparar electrodomésticos?
Respuesta: Las promociones están disponibles en nuestra página web.

Pregunta: ¿Cuál es el clima hoy?
Respuesta: El supermercado abre de 8:00 a 22:00 horas.

Pregunta: ¿Me pueden dar fiado?
Respuesta: Las promociones están disponibles en nuestra página web.

Pregunta: ¿

El valor de similitud mínima fue ajustado manualmente buscando un equilibrio entre aceptar preguntas válidas reformuladas y evitar respuestas incorrectas ante consultas fuera del dominio del supermercado.

Cuando el límite mínimo de similitud era demasiado alto, el chatbot rechazaba consultas válidas. En cambio, cuando era demasiado bajo, devolvía respuestas incorrectas para preguntas no relacionadas.

# TP5 - Chatbot RAG con LLM, embeddings y base de datos vectorial

En esta sección se resuelven las consignas del TP5 a partir del dataset de preguntas y respuestas creado en el TP4.

La aplicación sigue siendo un chatbot de atención al cliente para un supermercado.

## a) Creación del conjunto de datos de evaluación

Además del dataset original utilizado como base de conocimiento, se crea un dataset de evaluación con la misma lógica: preguntas y respuestas esperadas.

Este dataset sirve para probar si el chatbot recupera el contexto correcto y si responde de manera adecuada.

In [7]:
# Dataset de evaluación o prueba
# Cada elemento contiene:
# - pregunta: consulta nueva del usuario
# - respuesta_esperada: respuesta correcta esperada
# - id_contexto_esperado: índice del dataset original que debería recuperarse como contexto principal

dataset_evaluacion = [
    {
        "pregunta": "¿A qué hora abre el supermercado?",
        "respuesta_esperada": "El supermercado abre de 8:00 a 22:00 horas.",
        "id_contexto_esperado": 0
    },
    {
        "pregunta": "¿Puedo pagar con tarjeta?",
        "respuesta_esperada": "Sí, aceptamos todos los medios de pago.",
        "id_contexto_esperado": 1
    },
    {
        "pregunta": "¿Hacen envíos a domicilio?",
        "respuesta_esperada": "Sí, realizamos envíos a domicilio dentro de la ciudad.",
        "id_contexto_esperado": 3
    },
    {
        "pregunta": "¿Dónde veo las promociones?",
        "respuesta_esperada": "Las promociones están disponibles en nuestra página web.",
        "id_contexto_esperado": 5
    },
    {
        "pregunta": "¿Puedo cambiar algo que compré?",
        "respuesta_esperada": "Sí, puede cambiar productos con ticket de compra.",
        "id_contexto_esperado": 6
    },
    {
        "pregunta": "¿Aceptan Mercado Pago?",
        "respuesta_esperada": "Sí, aceptamos Mercado Pago.",
        "id_contexto_esperado": 7
    },
    {
        "pregunta": "¿Cuál es el teléfono de contacto?",
        "respuesta_esperada": "El teléfono es 351-1234567.",
        "id_contexto_esperado": 8
    },
    {
        "pregunta": "¿Abren los domingos?",
        "respuesta_esperada": "Sí, abrimos los domingos.",
        "id_contexto_esperado": 9
    },
    {
        "pregunta": "¿Tienen productos congelados?",
        "respuesta_esperada": "Sí, contamos con una sección de congelados.",
        "id_contexto_esperado": 10
    }
]

print("Cantidad de preguntas de evaluación:", len(dataset_evaluacion))


Cantidad de preguntas de evaluación: 9


## b) Elección del modelo LLM y de dos modelos de embeddings

### Modelo LLM elegido

Se eligió el modelo **google/flan-t5-small** de HuggingFace.

La elección se fundamenta en que se trata de un modelo ligero, capaz de ejecutarse en Google Colab sin demandar una gran cantidad de recursos computacionales. Además, está diseñado para seguir instrucciones, por lo que puede utilizar un contexto proporcionado junto con una pregunta para generar respuestas de manera adecuada.


### Modelos de embeddings elegidos

Se eligieron dos modelos de la librería Sentence Transformers:

1. **sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2**
2. **sentence-transformers/distiluse-base-multilingual-cased-v2**

Ambos modelos son multilingües, por lo tanto pueden trabajar con consultas en español. Además, generan embeddings semánticos, es decir que no dependen solamente de repetir palabras exactas como TF-IDF, sino que intentan representar el significado de las frases.

La comparación entre ambos permite observar cuál recupera mejor las preguntas y respuestas del dataset del supermercado.

In [8]:
# Instalación de librerías necesarias

!pip install -q sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.8 MB/s eta 0:00:00


In [9]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

## Preparación de la base de conocimiento

La base de conocimiento del chatbot será el dataset creado en el TP4. Cada documento combina una pregunta frecuente con su respuesta correspondiente.

Luego esos documentos serán convertidos a embeddings y almacenados en FAISS, que funcionará como base de datos vectorial.

In [10]:
preguntas = [item[0] for item in preguntas_respuestas]
respuestas = [item[1] for item in preguntas_respuestas]

base_conocimiento = []

for i, (pregunta, respuesta) in enumerate(preguntas_respuestas):
    documento = f"""Pregunta frecuente: {pregunta}
Respuesta: {respuesta}"""
    base_conocimiento.append({
        "id": i,
        "pregunta": pregunta,
        "respuesta": respuesta,
        "documento": documento
    })

print("Cantidad de documentos en la base de conocimiento:", len(base_conocimiento))
print(base_conocimiento[0])

Cantidad de documentos en la base de conocimiento: 21
{'id': 0, 'pregunta': '¿Cuál es el horario del supermercado?', 'respuesta': 'El supermercado abre de 8:00 a 22:00 horas.', 'documento': 'Pregunta frecuente: ¿Cuál es el horario del supermercado?\nRespuesta: El supermercado abre de 8:00 a 22:00 horas.'}


## c) Implementación de una clase ChatBot

La clase ChatBotRAG utiliza tres partes principales:

1. Un modelo de embeddings para transformar preguntas y documentos en vectores.
2. FAISS como base de datos vectorial para recuperar los contextos más parecidos.
3. Un modelo LLM de HuggingFace para generar una respuesta usando el contexto recuperado.

Este enfoque se llama RAG, porque primero recupera información relevante y luego genera una respuesta basada en esa información.

In [11]:
class ChatBotRAG:
    def __init__(self, base_conocimiento, modelo_embedding, modelo_llm="google/flan-t5-small", top_k=3):
        self.base_conocimiento = base_conocimiento
        self.modelo_embedding_nombre = modelo_embedding
        self.top_k = top_k
        self.max_new_tokens = 256 # Definir max_new_tokens como atributo de instancia

        # Carga del modelo de embeddings
        self.embedding_model = SentenceTransformer(modelo_embedding)

        # Carga del LLM de HuggingFace
        # Para modelos T5, pipeline('text-generation') no es compatible.
        # Cargamos el modelo y el tokenizador directamente para un control más preciso.
        self.tokenizer = AutoTokenizer.from_pretrained(modelo_llm)
        self.llm_model = AutoModelForSeq2SeqLM.from_pretrained(modelo_llm)

        # Se preparan los textos que se van a guardar en la base vectorial
        self.documentos = [item["documento"] for item in base_conocimiento]

        # Se calculan los embeddings de los documentos
        embeddings = self.embedding_model.encode(self.documentos, convert_to_numpy=True)
        embeddings = embeddings.astype("float32")

        # Normalizamos los vectores para usar similitud coseno con FAISS
        faiss.normalize_L2(embeddings)
        self.embeddings = embeddings

        # Creamos el índice FAISS
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(embeddings)

    def recuperar_contexto(self, pregunta_usuario):
        # Convertimos la pregunta del usuario en embedding
        vector_pregunta = self.embedding_model.encode([pregunta_usuario], convert_to_numpy=True).astype("float32")
        faiss.normalize_L2(vector_pregunta)

        # Buscamos los documentos más similares
        scores, indices = self.index.search(vector_pregunta, self.top_k)

        resultados = []
        for score, idx in zip(scores[0], indices[0]):
            item = self.base_conocimiento[idx].copy()
            item["score"] = float(score)
            resultados.append(item)

        return resultados

    def responder(self, pregunta_usuario):
        contextos = self.recuperar_contexto(pregunta_usuario)
        contexto_texto = "\n\n".join([c["documento"] for c in contextos])

        prompt_parts = [
            "Responde en español usando únicamente el contexto proporcionado.",
            "Si la respuesta no está en el contexto, responde: No tengo información suficiente para responder.",
            f"Contexto:\n{contexto_texto}",
            f"Pregunta del usuario:\n{pregunta_usuario}",
            "Respuesta:"
        ]
        prompt = "\n\n".join(prompt_parts)

        # Usar el modelo y tokenizador directamente para generar texto
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.llm_model.device) # Ensure inputs are on the same device as the model
        outputs = self.llm_model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        respuesta_raw = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # El LLM de HuggingFace para text-generation a menudo incluye el prompt en su salida.
        # Necesitamos extraer solo el texto generado. Reimplementamos la lógica de extracción robusta.
        if respuesta_raw.startswith(prompt):
            respuesta = respuesta_raw[len(prompt):].strip()
        else:
            # Fallback si por alguna razón el prompt no está al principio exacto.
            # Intentar encontrar 'Respuesta:' y tomar todo después de eso.
            search_marker = "Respuesta:"
            start_index = respuesta_raw.rfind(search_marker)
            if start_index != -1:
                respuesta = respuesta_raw[start_index + len(search_marker):].strip()
            else:
                respuesta = respuesta_raw.strip() # Fallback final

        return {
            "pregunta": pregunta_usuario,
            "respuesta": respuesta,
            "contextos": contextos,
            "modelo_embedding": self.modelo_embedding_nombre
        }

In [12]:
modelo_embedding_1 = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo_embedding_2 = "sentence-transformers/distiluse-base-multilingual-cased-v2"

chatbot_emb_1 = ChatBotRAG(
    base_conocimiento=base_conocimiento,
    modelo_embedding=modelo_embedding_1,
    top_k=3
)

chatbot_emb_2 = ChatBotRAG(
    base_conocimiento=base_conocimiento,
    modelo_embedding=modelo_embedding_2,
    top_k=3
)

# Asegurarse de que mejor_chatbot apunte a la instancia correcta y actualizada
mejor_chatbot = chatbot_emb_1

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [13]:
# Volvemos a evaluar el LLM con el chatbot reinicializado y el pipeline/extracción corregidos

def similitud_textos(modelo_embedding, texto_1, texto_2):
    vectores = modelo_embedding.encode([texto_1, texto_2], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(vectores)
    return float(np.dot(vectores[0], vectores[1]))


def evaluar_llm_simple(chatbot, dataset_evaluacion):
    filas = []

    for ejemplo in dataset_evaluacion:
        salida = chatbot.responder(ejemplo["pregunta"])
        respuesta_chatbot = salida["respuesta"]
        contexto_usado = salida["contextos"][0]["documento"]

        # Answer relevancy aproximado:
        # similitud entre la respuesta generada y la respuesta esperada
        answer_relevancy = similitud_textos(
            chatbot.embedding_model,
            respuesta_chatbot,
            ejemplo["respuesta_esperada"]
        )

        # Faithfulness aproximado:
        # similitud entre la respuesta generada y el contexto recuperado principal
        faithfulness = similitud_textos(
            chatbot.embedding_model,
            respuesta_chatbot,
            contexto_usado
        )

        filas.append({
            "pregunta": ejemplo["pregunta"],
            "respuesta_esperada": ejemplo["respuesta_esperada"],
            "respuesta_chatbot": respuesta_chatbot,
            "answer_relevancy_aprox": answer_relevancy,
            "faithfulness_aprox": faithfulness
        })

    return pd.DataFrame(filas)

metricas_llm_final_v5 = evaluar_llm_simple(mejor_chatbot, dataset_evaluacion)
metricas_llm_final_v5

,pregunta,respuesta_esperada,respuesta_chatbot,answer_relevancy_aprox,faithfulness_aprox
0,¿A qué hora abre el supermercado?,El supermercado abre de 8:00 a 22:00 horas.,El supermercado abre de 8:00 a 22:00 horas.,1.000000,0.853077
1,¿Puedo pagar con tarjeta?,"Sí, aceptamos todos los medios de pago.","S, algunas tarjetas tienen promociones en cuot...",0.253254,0.971189
2,¿Hacen envíos a domicilio?,"Sí, realizamos envíos a domicilio dentro de la...","S, realizamos envos a domicilio dentro de la c...",0.881185,0.825072
3,¿Dónde veo las promociones?,Las promociones están disponibles en nuestra p...,Question of the user: Where do we go?,0.145528,0.293502
4,¿Puedo cambiar algo que compré?,"Sí, puede cambiar productos con ticket de compra.","S, puede cambiar productos con ticket de compra.",0.938580,0.812385
5,¿Aceptan Mercado Pago?,"Sí, aceptamos Mercado Pago.","S, aceptamos Mercado Pago.",0.952784,0.762212
6,¿Cuál es el teléfono de contacto?,El teléfono es 351-1234567.,The teléfono es 351-1234567.,0.997515,0.617598
7,¿Abren los domingos?,"Sí, abrimos los domingos.","S, abrimos los domingos.",0.973349,0.855947
8,¿Tienen productos congelados?,"Sí, contamos con una sección de congelados.","S, contamos con una sección de congelados.",0.977138,0.852281


In [14]:
# Promedio general de métricas del LLM con respuestas limpias y pipeline/extracción corregidos
metricas_llm_final_v5[["answer_relevancy_aprox", "faithfulness_aprox"]].mean()

,0
answer_relevancy_aprox,0.791037
faithfulness_aprox,0.760362


## Creación de los chatbots con dos modelos de embeddings

Se crean dos versiones del mismo chatbot. La única diferencia entre ellas es el modelo de embeddings utilizado.

Esto permite comparar si cambia la calidad de recuperación de contexto y la respuesta final.

## d) Prueba del chatbot con el dataset de evaluación

Se prueba cada chatbot con las preguntas creadas en el dataset de evaluación.

Para comparar los modelos de embeddings, se observa principalmente si el contexto recuperado coincide con el esperado y si la respuesta generada es coherente con la respuesta esperada.

In [15]:
def evaluar_chatbot(chatbot, dataset_evaluacion):
    resultados = []

    for ejemplo in dataset_evaluacion:
        salida = chatbot.responder(ejemplo["pregunta"])
        contexto_top_1 = salida["contextos"][0]

        resultados.append({
            "modelo_embedding": salida["modelo_embedding"],
            "pregunta": ejemplo["pregunta"],
            "respuesta_esperada": ejemplo["respuesta_esperada"],
            "respuesta_chatbot": salida["respuesta"],
            "id_contexto_esperado": ejemplo["id_contexto_esperado"],
            "id_contexto_recuperado_top1": contexto_top_1["id"],
            "pregunta_recuperada_top1": contexto_top_1["pregunta"],
            "score_top1": contexto_top_1["score"],
            "recuperacion_correcta_top1": contexto_top_1["id"] == ejemplo["id_contexto_esperado"]
        })

    return pd.DataFrame(resultados)

resultados_emb_1 = evaluar_chatbot(chatbot_emb_1, dataset_evaluacion)
resultados_emb_2 = evaluar_chatbot(chatbot_emb_2, dataset_evaluacion)

resultados_comparacion = pd.concat([resultados_emb_1, resultados_emb_2], ignore_index=True)
resultados_comparacion

,modelo_embedding,pregunta,respuesta_esperada,respuesta_chatbot,id_contexto_esperado,id_contexto_recuperado_top1,pregunta_recuperada_top1,score_top1,recuperacion_correcta_top1
0,sentence-transformers/paraphrase-multilingual-...,¿A qué hora abre el supermercado?,El supermercado abre de 8:00 a 22:00 horas.,El supermercado abre de 8:00 a 22:00 horas.,0,0,¿Cuál es el horario del supermercado?,0.901331,True
1,sentence-transformers/paraphrase-multilingual-...,¿Puedo pagar con tarjeta?,"Sí, aceptamos todos los medios de pago.","S, algunas tarjetas tienen promociones en cuot...",1,12,¿Puedo pagar en cuotas?,0.608744,False
2,sentence-transformers/paraphrase-multilingual-...,¿Hacen envíos a domicilio?,"Sí, realizamos envíos a domicilio dentro de la...","S, realizamos envos a domicilio dentro de la c...",3,3,¿Tienen envío a domicilio?,0.815964,True
3,sentence-transformers/paraphrase-multilingual-...,¿Dónde veo las promociones?,Las promociones están disponibles en nuestra p...,Question of the user: Where do we go?,5,5,¿Dónde puedo consultar promociones?,0.750627,True
4,sentence-transformers/paraphrase-multilingual-...,¿Puedo cambiar algo que compré?,"Sí, puede cambiar productos con ticket de compra.","S, puede cambiar productos con ticket de compra.",6,6,¿Puedo cambiar un producto?,0.726674,True
5,sentence-transformers/paraphrase-multilingual-...,¿Aceptan Mercado Pago?,"Sí, aceptamos Mercado Pago.","S, aceptamos Mercado Pago.",7,7,¿Aceptan Mercado Pago?,0.819584,True
6,sentence-transformers/paraphrase-multilingual-...,¿Cuál es el teléfono de contacto?,El teléfono es 351-1234567.,The teléfono es 351-1234567.,8,8,¿Cuál es el teléfono del supermercado?,0.620729,True
7,sentence-transformers/paraphrase-multilingual-...,¿Abren los domingos?,"Sí, abrimos los domingos.","S, abrimos los domingos.",9,9,¿Tienen atención los domingos?,0.838998,True
8,sentence-transformers/paraphrase-multilingual-...,¿Tienen productos congelados?,"Sí, contamos con una sección de congelados.","S, contamos con una sección de congelados.",10,10,¿Venden productos congelados?,0.833530,True
9,sentence-transformers/distiluse-base-multiling...,¿A qué hora abre el supermercado?,El supermercado abre de 8:00 a 22:00 horas.,El supermercado abre de 8:00 a 22:00 horas.,0,0,¿Cuál es el horario del supermercado?,0.794183,True


In [16]:
# Resumen de recuperación correcta para cada modelo de embedding

resumen_recuperacion = resultados_comparacion.groupby("modelo_embedding")["recuperacion_correcta_top1"].mean().reset_index()
resumen_recuperacion["porcentaje_top1_correcto"] = resumen_recuperacion["recuperacion_correcta_top1"] * 100
resumen_recuperacion

,modelo_embedding,recuperacion_correcta_top1,porcentaje_top1_correcto
0,sentence-transformers/distiluse-base-multiling...,0.888889,88.888889
1,sentence-transformers/paraphrase-multilingual-...,0.888889,88.888889


## Conclusión de la comparación entre embeddings

Después de ejecutar las pruebas, se debe observar qué modelo recuperó más veces el contexto correcto.

El modelo que obtenga mayor porcentaje de recuperación correcta en el primer resultado será el más conveniente para esta aplicación, porque el chatbot depende mucho de recuperar bien la información antes de generar la respuesta.

Si ambos modelos obtienen resultados similares, se puede elegir el modelo más liviano o más rápido. En una aplicación simple de atención al cliente, la velocidad también es importante porque las respuestas deben generarse de manera rápida.